In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
!kaggle datasets download -d kunalgupta2616/flickr-8k-images-with-captions

In [ ]:
!unzip \*.zip && rm *.zip

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow import keras

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

image_dir = '/content/Images'
caption_file = '/content/captions.txt'

image_to_captions = {}

with open(caption_file, 'r') as f:
  lines = f.readlines()[1:]

for line in lines:
  line = line.strip()
  if not line:
    continue

  parts = line.split(',', 1)
  image_name = parts[0]
  caption_text = parts[1]

  # Add the caption to our dictionary
  if image_name not in image_to_captions:
    image_to_captions[image_name] = []

  image_to_captions[image_name].append(caption_text)

# --- Let's visualize the first image! ---
# Grab the very first image filename from our dictionary
sample_image_name = list(image_to_captions.keys())[3]
sample_image_path = os.path.join(image_dir, sample_image_name)

# Plot the image
img = mpimg.imread(sample_image_path)
plt.imshow(img)
plt.axis('off') # Hides the grid numbers
plt.show()

# Print the 5 captions underneath
print(f"File: {sample_image_name}\n")
print("Captions:")
for caption in image_to_captions[sample_image_name]:
    print(f"- {caption}")


Loading the resnet50 model now without fully connected layer

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

# Keras image processing tools
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# 1. Load the model
model = ResNet50(weights='imagenet', include_top=False, pooling='None')

image_dir = '/content/Images'
features = {}

# 2. Iterate through UNIQUE IMAGES (not 5 times same image)
unique_images = list(image_to_captions.keys())

print(f"Extracting features for {len(unique_images)} images...")

for image_name in tqdm(unique_images):
    image_path = os.path.join(image_dir, image_name)

    # load_img: Grabs the image and forces it to 224x224 pixels (ResNet's required size)
    image = load_img(image_path, target_size=(224, 224))

    # img_to_array: Converts the image pixels into a Numpy matrix of numbers
    image = img_to_array(image)

    # np.expand_dims: ResNet expects a "batch" of images, not just one.
    # This changes the shape from (224, 224, 3) to (1, 224, 224, 3)
    image = np.expand_dims(image, axis=0)

    # preprocess_input: Adjusts the image colors to match exactly how ResNet was originally trained
    image = preprocess_input(image)

    # 3. Pass the image through the model to get the feature vector
    feature_vector = model.predict(image, verbose=0)
    feature_vector = feature_vector.reshape((49, 2048))

    # Store it in our dictionary
    features[image_name] = feature_vector

# 4. Saving the dictionary to colab workspace using modern NumPy compression
np.savez_compressed('features.npz', **features)

print("All features extracted and compressed into 'features.npz'!")

In [ ]:
import string
import json
from tensorflow.keras.preprocessing.text import Tokenizer

# 1. Clean the captions and add sequence tokens
clean_captions_list = [] # A flat list of all captions to feed the tokenizer
clean_mapping = {}       # A dictionary mapping image names to their cleaned captions

print("Cleaning captions")

for image_name, captions_list in image_to_captions.items():
  clean_mapping[image_name] = []

  for caption in captions_list:
    # convert to lowercasr
    caption = caption.lower()

    # remove punctuation
    caption = caption.translate(str.maketrans('', '', string.punctuation))

    # remove extra spaces
    caption = " ".join(caption.split())

    # add the start and end token
    caption = f"startseq {caption} endseq"

    # Save to our new structures
    clean_mapping[image_name].append(caption)
    clean_captions_list.append(caption)

# 2. Tokenise the text
print("Tokenizing vocabulary...")
tokenizer = Tokenizer()
# This reads all 40,000 captions, finds unique words, and numbers them
tokenizer.fit_on_texts(clean_captions_list)

# We add +1 to the vocab size for the "0" padding we will add later
vocab_size = len(tokenizer.word_index) + 1

# Calculate the length of the longest caption (needed for padding)
max_length = max(len(caption.split()) for caption in clean_captions_list)

print(f"Total Unique Words (Vocabulary Size): {vocab_size}")
print(f"Longest Caption (Max Length): {max_length} words")

# 3. Save everything using modern JSON for training and deployment
# Convert the tokenizer to a JSON string and save
with open('tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())

# Save the cleaned captions dictionary
with open('clean_captions.json', 'w', encoding='utf-8') as f:
    json.dump(clean_mapping, f)

# Export the max_length config for your Streamlit web app
with open('model_config.json', 'w', encoding='utf-8') as f:
    json.dump({'max_length': max_length}, f)

print("Text processing complete and universally saved as JSON!")

Designing the Model Architecture

In [ ]:
from tensorflow.keras.layers import ( Input, Dense, LSTM, Embedding, Dropout, Concatenate, TimeDistributed, AdditiveAttention )
from tensorflow.keras.models import Model

# 1. Image Feature Pathway (Encoder)
# Now expects the full (49, 2048) spatial grid, not a single pooled vector
inputs1 = Input(shape=(49, 2048), name='Image_features')
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)          # (49, 256) -- projected image regions

# 2. Text Sequence Pathway (Decoder)
inputs2 = Input(shape=(max_length,), name='Text_sequence')
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(256, return_sequences=True, use_cudnn=False)(se2)       # (max_length, 256) -- a query for EVERY word, not just the last

# 3. Attention -- for each word, look across all 49 image regions
context = AdditiveAttention()([se3, fe2])         # se3 = query, fe2 = value/key -> (max_length, 256)

# 4. Merge text and attended image context at every timestep
decoder1 = Concatenate()([se3, context])          # (max_length, 512)
decoder2 = TimeDistributed(Dense(256, activation='relu'))(decoder1)

# 5. Final Output Layer -- a word prediction at every position
outputs = TimeDistributed(Dense(vocab_size, activation='softmax'), name="Next_Word_Prediction")(decoder2)

# 6. Tie it together
model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

model.summary()

Training the above model architecture

In [ ]:
import json
import numpy as np
from tensorflow.keras.preprocessing.text import tokenizer_from_json

# 1. Load compressed image features natively using NumPy
features = np.load('features.npz')

# 2. Load cleaned captions from the universally readable JSON format
with open('clean_captions.json', 'r', encoding='utf-8') as f:
    clean_mapping = json.load(f)

# 3. Reconstruct the Keras tokenizer from the JSON string
with open('tokenizer.json', 'r', encoding='utf-8') as f:
    tokenizer_json_string = f.read()
    tokenizer = tokenizer_from_json(tokenizer_json_string)

# 4. Calculate Vocab Size & Max Length dynamically
vocab_size = len(tokenizer.word_index) + 1

all_captions = []
for key in clean_mapping:
    for cap in clean_mapping[key]:
        all_captions.append(cap)

max_length = max(len(caption.split()) for caption in all_captions)

print(f"Features loaded: {len(features)} images")
print(f"Clean captions loaded: {len(clean_mapping)} images")
print(f"Vocabulary Size: {vocab_size}")
print(f"Max Caption Length: {max_length}")

Building Data Generator for training

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def seq2seq_data_generator(clean_mapping, features, tokenizer, max_length, batch_size):
    X1, X2, y = list(), list(), list()

    while True:
        for image_name, captions in clean_mapping.items():
            # Skip if missing
            if image_name not in features:
                continue

            # Fetch the new spatial grid features and ensure correct shape
            img_feat = features[image_name]
            if img_feat.shape != (49, 2048):
                img_feat = img_feat.reshape(49, 2048)

            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]

                # A valid sequence needs at least a start and end token
                if len(seq) < 2:
                    continue

                # --- THE BIG CHANGE: SEQUENCE SHIFTING ---
                # Input sees everything EXCEPT the last word
                in_seq = seq[:-1]
                # Target output is everything EXCEPT the first word
                out_seq = seq[1:]

                # Right-pad both sequences to your max_length
                in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                out_seq = pad_sequences([out_seq], maxlen=max_length, padding='post')[0]

                X1.append(img_feat)
                X2.append(in_seq)
                y.append(out_seq)

                # Yield when the batch is full
                if len(X1) == batch_size:
                    yield {
                        'Image_features': np.array(X1),
                        'Text_sequence': np.array(X2)
                    }, np.array(y)

                    X1, X2, y = list(), list(), list()

# Instantiate the new generator
batch_size = 64
train_generator = seq2seq_data_generator(
    clean_mapping, features, tokenizer, max_length, batch_size
)

print("Seq2Seq Attention Generator is ready!")

In [ ]:
# Calculate exact steps per epoch based on total captions
total_captions = sum(len(caps) for caps in clean_mapping.values())
steps_per_epoch = total_captions // batch_size
epochs = 20

print(f"Total captions : {total_captions}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Starting attention model training for {epochs} epochs...")
print(f"Total steps per epoch: {steps_per_epoch}")

history = model.fit(
    train_generator,
    epochs=epochs,
    steps_per_epoch=steps_per_epoch,
    verbose=1
)

# Save this new advanced model using the modern Keras v3 standard!
model.save('attention_caption_model.keras')
print("Training complete! Weights saved as .keras.")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

# Initialize ResNet WITHOUT pooling
resnet_model = ResNet50(weights='imagenet', include_top=False)
index_to_word = {index: word for word, index in tokenizer.word_index.items()}

def generate_greedy_caption(image_path, caption_model, max_length, tokenizer, index_to_word):
    # 1. Process new image
    image = load_img(image_path, target_size=(224, 224))
    image_pixels = img_to_array(image)
    image_pixels = np.expand_dims(image_pixels, axis=0)
    image_pixels = preprocess_input(image_pixels)

    # 2. Extract spatial features and reshape to (1, 49, 2048)
    image_feature = resnet_model.predict(image_pixels, verbose=0)
    image_feature = image_feature.reshape(1, 49, 2048)

    # 3. Setup Tokens and Initial Sequence
    start_id = tokenizer.word_index['startseq']
    end_id = tokenizer.word_index['endseq']

    # Using int32 to prevent graph execution crashes!
    sequence = np.zeros((1, max_length), dtype=np.int32)
    sequence[0, 0] = start_id

    predicted_words = []

    # 4. Simple Greedy Autoregressive Loop
    for i in range(max_length - 1):
        # Predict the distribution for all timesteps at once
        yhat = caption_model.predict({
            'Image_features': image_feature,
            'Text_sequence': sequence
        }, verbose=0)

        # Get the absolute most likely word at the current timestep
        next_word_id = np.argmax(yhat[0, i, :])

        # Stop if the model predicts the end token
        if next_word_id == end_id:
            break

        # Write the word into the sequence for the next loop
        sequence[0, i + 1] = next_word_id

        # Translate to English
        word = index_to_word.get(next_word_id, '')
        if word:
            predicted_words.append(word)

    return ' '.join(predicted_words)

print("Greedy Search Attention engine ready!")

In [ ]:
import matplotlib.pyplot as plt

# Extract the loss values from the training history
loss = history.history['loss']
epochs_range = range(1, len(loss) + 1)

# Set up the visual styling
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, loss, marker='o', linestyle='-', color='#1f77b4', linewidth=2, label='Training Loss')

# Add labels and title
plt.title('Attention Model Training: Loss vs. Epoch', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (Categorical Crossentropy)', fontsize=12)

# Make it look professional
plt.grid(True, linestyle='-', alpha=0.7)
plt.legend(fontsize=12)

# Save the plot as an image so you can show it off later!
plt.savefig('loss_curve.png', bbox_inches='tight')

# Display it in Colab
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import os

# Define a list of your test image paths (make sure these are uploaded to Colab)
# Alternatively, if they are in a folder, you can use os.listdir()
test_images = [
    '/content/vizsla-running-022321.jpg',
    '/content/images (4).jpg',
    '/content/images (5).jpg',
    '/content/Images/1003163366_44323f5815.jpg',
    '/content/Images/1000268201_693b08cb0e.jpg',
    '/content/images (6).jpg'
]

print("Starting Batch Inference...\n" + "="*40)

for img_path in test_images:
    # Check if file exists to prevent crashing
    if not os.path.exists(img_path):
        print(f"⚠️ Could not find {img_path}. Skipping...")
        continue

    # 1. Generate the caption using your new Greedy Search function
    caption = generate_greedy_caption(img_path, model, max_length, tokenizer, index_to_word)

    # 2. Load the image purely for visual display
    display_img = load_img(img_path)

    # 3. Plot the image and the caption
    plt.figure(figsize=(6, 6))
    plt.imshow(display_img)
    plt.axis('off')

    # Wrap the title text so long captions don't get cut off
    plt.title(caption, fontsize=14, color='darkblue', wrap=True)
    plt.show()

    print(f"File: {img_path}")
    print(f"Prediction: {caption}")
    print("-" * 50)

In [ ]:
from google.colab import drive
import shutil
import os

# 1. Prompt to log in and connect your Google Drive
drive.mount('/content/drive')

# 2. Define ALL the modernized files we generated today
files_to_export = [
    'attention_caption_model.keras', # Modern model
    'tokenizer.json',                # Modern tokenizer
    'model_config.json',             # Streamlit config
    'loss_curve.png',                # Your training graph
    'features.npz',                  # Compressed image features (backup)
    'clean_captions.json'            # Cleaned text mapping (backup)
]

# 3. Create a dedicated folder in your Drive to keep things clean
backup_folder = '/content/drive/MyDrive/Image_Captioning_Backup'
os.makedirs(backup_folder, exist_ok=True)

# 4. Copy everything safely to Drive
for file_name in files_to_export:
    if os.path.exists(file_name):
        destination = os.path.join(backup_folder, file_name)
        shutil.copy(file_name, destination)
        print(f"✅ Successfully backed up {file_name}!")
    else:
        print(f"⚠️ Could not find {file_name}. Did it save correctly?")